In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
fedepsim = h5py.File("/home/yousen/Public/ndlar_shared/data/tred_2x2_2025010/MiniRun5_1E19_RHC.convert2h5.0000000.EDEPSIM.hdf5")

In [ ]:
fedepsim.keys()

In [ ]:
fedepsim['segments']['pdg_id']

In [ ]:
fedepsim['segments']['dEdx']
fedepsim['segments']['dE']

In [ ]:
segments = pd.DataFrame.from_records(fedepsim['segments'][:])

In [ ]:
segments['dx'] = segments['dE'] / segments['dEdx']
segments['Ne'] = birks(segments['dE'], segments['dEdx'], 0.5, 1.38)

In [ ]:
mu_segs = segments[np.abs(segments['pdg_id']) == 13]

In [ ]:
mu_segs['dx'].plot(kind='hist', bins=200)

In [ ]:
for i in [0.5, 1, 2, 3, 4, 5]:
    print('<', i, 'cm', np.sum(mu_segs['dx']<i) / len(mu_segs['dx']))
mu_segs['dx'].plot(kind='hist', bins=200, cumulative=True, density=True)
plt.grid(True)

In [ ]:
mu_segs['dx'].plot(kind='hist', bins=200, density=True, weights=mu_segs['dE'])

In [ ]:
mu_segs['dx'].plot(kind='hist', bins=200, cumulative=True, density=True, weights=mu_segs['dE'])
plt.grid(True)

In [ ]:
H, edges = np.histogram(mu_segs['dx'], weights=mu_segs['dE'], range=(0, 100), bins=200)
cH = np.cumsum(H)/np.sum(H)
for i in [0.5, 1, 2, 3, 4, 5]:
    for j in range(0, len(H)):
        if abs(edges[j+1]-i)<1E-4:
            print(f'< {i} cm, fraction  {cH[j]}')

In [ ]:
# Define dx ranges
# bins = [(0, 0.1), (0.1, 0.5), (0.5, 1.0), (1.0, 5.0), (5.0, 10.0), (10.0, 20.0)]
# bins = [(0, 0.1), (0.1, 0.5), (0.5, 1.0), (1.0, 5.0), (5.0, 10.0)]
# bins = [(0.5, 1.0), (1.0, 2.0), (2.0, 3.0), (3.0, 4.0)]
# bins = [(0.5, 1.0), (0.5, 1.0), (1.0, 2.0)]
bins = [(0, 0.5), (0.5, 1.0), (1.0, 2.0), (2.0, 30)]
# Create a plot
plt.figure(figsize=(10, 6))

# Plot each range
for low, high in bins:
    subset = mu_segs[(mu_segs['dx'] > low) & (mu_segs['dx'] <= high)]
    subset['dEdx'].hist(bins=100, range=(0, 10), alpha=0.5, label=f'{low} < dx < {high}, mean = {np.mean(subset["dEdx"])}', density=True)

# Customize plot
plt.xlabel('dEdx')
plt.ylabel('Frequency')
plt.title('Overlayed Histograms of dEdx by dx Range')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
for i in [0.5, 1, 2, 3, 4, 5]:
    print(i, np.sum(mu_segs['dx']<i) / len(mu_segs['dx']))

In [ ]:
def birks(dE, dEdx,
          efield: float, rho: float,
          A3t: float = 0.8, k3t: float = 0.0486,
          Wi:float = 23.6E-6):
    '''
    Return the number of electrons for a given energy deposition, under birks model.
    Reference: https://lar.bnl.gov/properties/pass.html
    Args:
        dE: 1D tensor of energy deposition, (npt,)
        dEdx: 1D tensor of dE/dx of the energy deposition (npt,)
        efield: electric field, e.g., 0.5 kV/cm
        rho: density of liquid argon, e.g., 1.38 g/cm^3
        A3t: default to 0.8
        k3t: default to 0.0486 (g/MeV cm^2) (kV/cm)
        Wi: W-value for ionization, default to 23.6E-6 MeV/pair
    Return:
        Q: 1D tensor of number of electrons after recombination

    Users are responsible to pass in arguments with consistent units.
    '''
    R = A3t / (1 + dEdx * (k3t / (efield * rho)))
    return R * dE / Wi

In [ ]:
# Define dx ranges
# bins = [(0, 0.1), (0.1, 0.5), (0.5, 1.0), (1.0, 5.0), (5.0, 10.0), (10.0, 20.0)]
# bins = [(0, 0.1), (0.1, 0.5), (0.5, 1.0), (1.0, 5.0), (5.0, 10.0)]
# bins = [(0.5, 1.0), (1.0, 2.0), (2.0, 3.0), (3.0, 4.0)]
bins = [(0., 0.5), (0.5, 1.0), (1.0, 2.0), (2.0, 30.0)]
# bins = [(0, 0.5)]
# Create a plot
plt.figure(figsize=(10, 6))

# Plot each range
for low, high in bins:
    subset = mu_segs[(mu_segs['dx'] > low) & (mu_segs['dx'] <= high)]
    dQ = subset['Ne']/subset['dx'] * 0.443
    plt.hist(dQ, bins=50, range=(0, 50_000), alpha=0.5, label=f'{low} < dx < {high}, mean = {np.mean(dQ)}', density=True)

# Customize plot
plt.xlabel('dQ')
plt.ylabel('Frequency')
plt.xlim(0, 50_000)
plt.title('Overlayed Histograms of dQ @ 4.43mm by dx Range')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
dQ = mu_segs['Ne']/mu_segs['dx']*0.443
dQw = mu_segs['dx']/0.443
plt.hist(dQ, weights=dQw, density=True, range=(0, 50_000), bins=50)
plt.title('weighted by dx/0.443')
plt.xlabel('dQ')
plt.grid(True)

In [ ]:
# Define dx ranges
bins = [(0., 0.5), (0.5, 1.0), (1.0, 2.0), (2.0, 30.0)]
# Create a plot
plt.figure(figsize=(10, 6))

# Plot each range
for low, high in bins:
    subset = mu_segs[(mu_segs['dx'] > low) & (mu_segs['dx'] <= high)]
    dQ = subset['Ne']/subset['dx'] * 0.443
    dQw = subset['dx'] / 0.443
    plt.hist(dQ, bins=50, range=(0, 50_000), alpha=0.5, label=f'{low} < dx < {high}, mean = {np.mean(dQ)}', density=True, weights=dQw)

# Customize plot
plt.xlabel('dQ')
plt.ylabel('Frequency')
plt.xlim(0, 50_000)
plt.title('Overlayed Histograms of dQ @ 4.43mm by dx Range; weighted by dx/0.443')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()